In [ ]:
%%capture

# Core training libraries
!pip install \
    transformers==4.44.2 \
    datasets==2.20.0 \
    tokenizers==0.19.1 \
    accelerate==0.34.2 \
    peft==0.12.0 \
    trl==0.9.6 \
    bitsandbytes==0.43.1 \
    evaluate==0.4.2

# Utilities
!pip install \
    numpy \
    pandas \
    scikit-learn \
    rich \
    pyyaml \
    python-dotenv \
    tqdm

# Evaluation (requires pydantic v2)
!pip install --upgrade pydantic
!pip install google-genai rouge-score

!pip uninstall -y deepspeed 2>/dev/null || true
!pip upgrade numpy
!pip upgrade torch

print("Installation complete!")
print("All dependencies compatible (pydantic v2 + google-genai)")

In [ ]:
import os

with open('env','w') as f:
  f.write('GOOGLE_API_KEY=\n')
  f.write('HF_TOKEN=\n')

print(".env file is created")

from dotenv import load_dotenv
load_dotenv()

try:
  with open('env','r') as f:
    print("keys in .env : ")
    print("="*60)
    for line in f:
      line = line.strip()
      if line and not line.startwith("#"):
        key = line.split('=')[0] # split each line from the = and take the first element
        value_preview = line.split('=')[1][10] + "........" if '=' in line else ""
        print(f"{key} : {value_preview}")
    print("="*60)
except FileNotFoundError:
  print(".env not found")

In [ ]:
import sys
import torch

print("="*60)
print("Environent check")
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")   


if(torch.cuda.is_available()):
  print(f"CUDA version: {torch.version.cuda}")
  print(f"cuDNN version: {torch.backends.cudnn.version()}")
  print(f"GPU Name: {torch.cuda.get_device_name(0)}")
  print(f"Device capability: {torch.cuda.get_device_capability(0)}")
  print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("No GPU detected. Training will be slow on CPU.")
print("="*60)
  

In [ ]:
import os
import random
import numpy as np
import torch


SEED =42
os.environ['PYTHONHASHSEED'] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
print(f"Random seed set to {SEED} for reproducibility.")

In [ ]:
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
!hf auth login --token $HF_TOKEN

print("Authentication successful with Hugging Face Hub!")

In [ ]:
import torch
from pprint import pprint

#checking the GPU dynamic compute dtype
# check if GPU supports bf16 (Ampere or later)
use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
print(f"Using bf16: {use_bf16}")

compute_dtype = torch.bfloat16 if use_bf16 else torch.float16
print(f"Selected compute dtype: {compute_dtype}")

CONFIG ={
    "base_model": "Qwen/Qwen2.5-1.5B-Instruct",
    
    "dataset_name": "lavita/AlpaCare-MedInstruct-52k",
    "dataset_split": "train",
    "dataset_subsample": 500,
    "train_val_split": 0.9,
    
    "max_length": 512,
    
    "num_train_epochs": 3,
    "max_steps": 250,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 64,
    "learning_rate": 2e-5,
    "warmup_ratio": 0.03,
    "warmup_steps": 10,
    "logging_steps": 10,
    "save_steps": 50,
    "eval_steps": 50,
    "save_total_limit": 2,
    
    # LoRA
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "lora_target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],

    # Quantization
    "load_in_4bit": True,
    "bnb_4bit_compute_dtype": compute_dtype,
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_quant": True,

    # Output
    "output_dir": "outputs/adapter",
    "push_to_hub": False,

    # Generation
    "max_new_tokens": 128,
    "temperature": 0.0,  # Deterministic
    "do_sample": True,

    # HF credentials
    'hf_username': '',
    'hub_model_name': '',
    
}

effective_batch_size = CONFIG["per_device_train_batch_size"] * CONFIG["gradient_accumulation_steps"]

print("="*60)
print("CONFIGURATION (COLAB FREE TIER)")
print("="*60)
pprint(CONFIG)
print("="*60)
print(f"Compute dtype: {compute_dtype}")
print(f"Using BF16: {use_bf16}")
print(f"Effective batch size: {effective_batch_size}")
print("="*60)

In [ ]:
from datasets import load_dataset, Dataset
import json

def load_my_dataset(dataset_name,split,subsample, seed =42):
    try:
        print(f"Loading dataset: {dataset_name}")
        dataset = load_dataset(dataset_name, split=split)
        dataset = dataset.shuffle(seed=seed).select(range(subsample))
        print(f"Dataset loaded and subsampled to {subsample} examples.")
        
    except Exception as e:
        print(f"Error loading dataset: {e}")
        print("Creating synthetic dataset for demonstration")